#### 基于Qwen2.5-0.5B 基础预训练模型和LoRA微调的的用户交易意图分析分类微调模型

##### 加载Qwen2.5-0.5B模型
 - 使用ModelScope对Transformers包的封装加载Transformers的AutoModel和AutoTokenizer;
 - 分别加载模型和分词器
##### 调整模型输出层
- 添加一个分类层
- 维度为896、in_features=896、out_features=2

In [1]:
from modelscope import AutoModelForSequenceClassification, AutoTokenizer

model_path = "Qwen/Qwen2.5-0.5B"

model = (AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path=model_path, num_labels=2)).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_path)

2026-03-11 11:17:58,268 - modelscope - INFO - Target directory already exists, skipping creation.
2026-03-11 11:18:00.320303: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-11 11:18:01.533358: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-11 11:18:05.234099: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OP

2026-03-11 11:18:10,799 - modelscope - INFO - Target directory already exists, skipping creation.


##### 模型测试
- 使用cuda0设备、token IDS、res_token_ids转PyTorch Tensor

In [ ]:
token_ids = tokenizer(["这个活动的后续流程是怎样的？"], return_tensors="pt", padding=True, truncation=True).to("cuda")
token_ids.input_ids[-1, :]
res_token_ids = model(**token_ids)
res_token_ids

151643

##### 初始化LoRA微调
预计微调500万参数，会自动冻结除target_modules以外的层
- 注意安装PEFT包（Powered By Transformers）；
- 初始化LoraConfig，包含任务类型（因果语言模型，即Transformers）；
- 设置rank以及缩放、设置暂退率为0.1同时配置需要融合LoRA层的原始模型层（应用到全部线性层）
- 创建LoRA Layer、LoRA With Linear、replace_linear_with_lora

In [50]:
from peft import LoraConfig, get_peft_model, TaskType

lora_conf = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "score"],
    inference_mode=False # 表示当前是训练模式，若为True则是在推理模式下的一些特定优化会被应用
)

trading_intent_classification_model = get_peft_model(model=model, peft_config=lora_conf) # 会自动冻结除target_modules以外的层
trading_intent_classification_model.print_trainable_parameters()

/miniconda/envs/d2l/lib/python3.9/site-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/miniconda/envs/d2l/lib/python3.9/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 8,800,000 || all params: 502,834,560 || trainable%: 1.7501


##### 加载数据集
- 分批加载训练集、验证集、测试集
- 数据预处理：对数据进行padding或truncation并将数据文本部分分别进行token化

In [123]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files={"train": "trading_intent_classification_lora/lora_adapter_example/classification_lora_train.csv", "val": "trading_intent_classification_lora/lora_adapter_example/classification_lora_val.csv", "test": "trading_intent_classification_lora/lora_adapter_example/classification_lora_test.csv"})

tokenized_datasets = dataset.map(
    function=lambda examples: tokenizer(examples['Text'], truncation=True, max_length=512, padding=True),
    batched=True,
    remove_columns=['Text']
)

tokenized_datasets = tokenized_datasets.rename_column("Label", "labels")

##### 配置训练参数
配置的训练参数包含以下内容：
- 设置模型训练检查点、结果的保存路径，已经覆盖已存在的输出目录；
- 需要训练过程中执行训练和评估；
- 超参设置，包括迭代次数、学习率、训练和验证集的批次大小、权重衰减系数；
- 设置模型的执行策略，包括每次epoch结束后进行一次验证、每次epoch结束后保存检查点、检查点的默认保存策略为每次epoch结束后；

In [ ]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score
import numpy as np
import torch

trading_intent_classification_model.config.pad_token_id = tokenizer.pad_token_id

training_args = TrainingArguments(
    output_dir="./trading_intent_classification_lora", # 模型训练检查点、成果的保存路径
    overwrite_output_dir=True,                         # 覆盖已存在的输出目录
    do_train=True,                                     # 是否执行训练
    do_eval=True,                                      # 是否执行评估
    learning_rate=2e-5,                                # 学习率
    per_device_train_batch_size=10,                    # 训练集批次大小
    per_device_eval_batch_size=10,                     # 验证集批次大小
    num_train_epochs=3,                                # 迭代次数
    weight_decay=0.01,                                 # 权重衰减系数
    eval_strategy="epoch",                             # 验证过程在每次epoch结束后执行
    save_strategy="epoch",                             # 在每次epoch结束后保存checkpoint
    metric_for_best_model="accuracy",                  # 选择最佳模型的指标（准确率）
    greater_is_better=True                             # 评估指标越大越好（True）
)

def compute_metrics(eval_pred) -> dict[str, float]:
    """模型评估指标计算

    Args:
        eval_pred (_type_): 模型训练输出

    Returns:
        dict[str, float]: 模型指标字典
    """
    
    logits, labels = eval_pred
    predictions = np.argmax(a=logits, axis=-1)
    
    return {"accuracy": accuracy_score(labels, predictions)}

trainer = Trainer(
    model=trading_intent_classification_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors="pt"),
    compute_metrics=compute_metrics
)

##### 开始训练模型
- 需要设置train的resume_from_checkpoint=True，使训练可自动加载ouput_dir中的最新检查点
- 注意：如果一开始没有检查点，则无法进行训练，需要先开启训练，后续有加载需要时再配置resume_from_checkpoint参数

In [127]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.116150,0.960000
2,No log,0.118864,0.960000
3,No log,0.117125,0.960000


TrainOutput(global_step=30, training_loss=0.0631349245707194, metrics={'train_runtime': 454.3518, 'train_samples_per_second': 0.66, 'train_steps_per_second': 0.066, 'total_flos': 57425204966400.0, 'train_loss': 0.0631349245707194, 'epoch': 3.0})

##### 模型测试
使用测试集中的数据测试模型的准确率

In [ ]:
token_ids = tokenizer([
        "Your unique user ID is 1172. For removal send STOP to 87239 customer services 08708034412",
        "Sorry . I will be able to get to you. See you in the morning.",
        "I can't right this second, gotta hit people up first",
        "Ur balance is now £500. Ur next question is: Who sang 'Uptown Girl' in the 80's ? 2 answer txt ur ANSWER to 83600. Good luck!",
        "Sorry, I'll call you  later. I am in meeting sir.",
        "WIN a £200 Shopping spree every WEEK Starting NOW. 2 play text STORE to 88039. SkilGme. TsCs08714740323 1Winawk! age16 £1.50perweeksub."
    ], 
    return_tensors="pt", padding=True, truncation=True).to("cuda")

trading_intent_classification_model.eval()
torch.argmax(input=trading_intent_classification_model(**token_ids).logits, dim=-1)

tensor([1, 0, 0, 1, 0, 1], device='cuda:0')

##### 保存和加载模型
- 使用model.save_pretrained(save_path)保存模型
- 使用tokenizer.save_pretrained(save_path)保存对应的分词器
- 使用PeftModel.from_pretrained(base_model, save_path)加载并使用

In [ ]:
from peft import PeftModel

save_directory = "./trading_intent_classification_lora/lora_adapter_example"

trading_intent_classification_model.save_pretrained(save_directory=save_directory)
tokenizer.save_pretrained(save_directory=save_directory)
trained_model = PeftModel.from_pretrained(model, save_directory)